# 0 — CLIP similarity pathways

This notebook introduces **four distinct operations** built from the same CLIP model:

1. **Image → text:** rank candidate descriptions for one image.
2. **Text → images:** rank candidate images for one prompt.
3. **Text → text:** compare normalized text embeddings.
4. **Image → images:** compare normalized image embeddings.

The uploaded notebook placed model loading, plotting, callbacks, and the entire Gradio app in one 450-line cell. Here those concerns are separated so each pathway is easier to inspect and reuse.

## Install and repository setup

Run this notebook from the repository after installing the editable package. CLIP inference works on CPU, MPS, or CUDA; a GPU is helpful but not required.

In [ ]:
# Uncomment once in a fresh environment.
# %pip install -e "..[ui,notebooks]"

In [ ]:
from clip_token_lab.clip import ClipEmbedder
from clip_token_lab.io import load_rgb_image

clip = ClipEmbedder()  # Downloads openai/clip-vit-base-patch32 on first use.
print("device:", clip.device)

## How the scores differ

For **image↔text**, CLIP returns scaled cross-modal logits. Applying softmax makes the scores sum to one **over the candidates supplied in that call**. Adding or removing a candidate changes every probability.

For **text↔text** and **image↔image**, the package normalizes embeddings and computes their dot product, which is cosine similarity. Negative values are retained; they are not probabilities.

## Pathway A — image → text

The image is processed once and compared with every text candidate in a shared CLIP embedding space.

In [ ]:
# image = load_rgb_image("path/to/image.jpg")
# candidates = ["a dog", "a cat", "a car", "a landscape"]
# for item in clip.image_to_text(image, candidates):
#     print(f"{item.score:8.2%}  {item.label}")

## Pathway B — text → images

This is the transpose of the first pathway: one prompt is ranked against a batch of images.

In [ ]:
# images = [load_rgb_image(path) for path in ["one.jpg", "two.jpg", "three.jpg"]]
# for item in clip.text_to_images("a dog playing outside", images):
#     print(f"{item.score:8.2%}  {item.label}")

## Pathway C — text → text

CLIP was trained contrastively across text and images, not as a dedicated sentence-embedding model. This pathway is useful for exploration, but a sentence-transformer may be a better default for production semantic search.

In [ ]:
reference = "a happy dog running through grass"
candidates = [
    "a dog playing outside",
    "a cat sleeping on a couch",
    "a car driving on a road",
]
clip.text_to_text(reference, candidates)

## Pathway D — image → images

Images are independently embedded, normalized, and compared with cosine similarity.

In [ ]:
# reference = load_rgb_image("reference.jpg")
# candidates = [load_rgb_image("one.jpg"), load_rgb_image("two.jpg")]
# clip.image_to_images(reference, candidates)

## Launch the complete UI

The app keeps model creation lazy and does not create a public share link unless explicitly requested.

In [ ]:
from clip_token_lab.apps.clip_intro import build_demo

demo = build_demo()
demo.launch(inline=True)

## Minimal scripts

- `scripts/clip/image_to_text.py`
- `scripts/clip/text_to_images.py`
- `scripts/clip/text_to_text.py`
- `scripts/clip/image_to_images.py`

Each script contains only argument parsing, image/text loading, one package call, and result printing.